### 1: Завдання

Завантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset 

In [1]:
import pandas as pd
import numpy as np
import timeit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from IPython.display import display

df = pd.read_csv('household_power_consumption.txt', sep=';', na_values='?', low_memory=False)
print('Датасет завантажено')

Датасет завантажено


### 2: Завдання

Здійснити data cleaning

In [2]:
def data_cleaning(df):
    df = df.dropna()
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    df = df.dropna()
    df = df.reset_index(drop=True)
    return df

df = data_cleaning(df)
print("Розмір:", len(df))
display(df.head())

Розмір: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### 3: Завдання

3.1. Окремими функціями сформувати вибірки (Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт)

In [5]:
def get_global_active_records(df):
    filtered_df = df[df['Global_active_power'] > 5]
    return filtered_df

display(get_global_active_records(df).head())
time_for_execution = timeit.timeit(lambda: get_global_active_records(df), number=1)
print(f'Час виконання функції: {time_for_execution} секунд')

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


Час виконання функції: 0.0040123000217135996 секунд


3.2. Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер

In [6]:
def get_global_intensity_records(df):
    filtered_df = df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20) & (df['Sub_metering_2'] > df['Sub_metering_3'])]
    return filtered_df

display(get_global_intensity_records(df).head()) 
time_for_execution = timeit.timeit(lambda: get_global_intensity_records(df), number=1)
print(f'Час виконання функції: {time_for_execution} секунд')

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


Час виконання функції: 0.010006000025896356 секунд


3.3. Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [10]:
def get_random_stats(df):
    random_stats = df.sample(n=500000, replace=False)
    stats = {
        'Sub_metering_1_mean': random_stats['Sub_metering_1'].mean(),
        'Sub_metering_2_mean': random_stats['Sub_metering_2'].mean(),
        'Sub_metering_3_mean': random_stats['Sub_metering_3'].mean()
    }
    return pd.DataFrame([stats])

display(get_random_stats(df).head()) 
time_for_execution = timeit.timeit(lambda: get_random_stats(df), number=1)
print(f'Час виконання функції: {time_for_execution} секунд')

,Sub_metering_1_mean,Sub_metering_2_mean,Sub_metering_3_mean
0,1.125308,1.297902,6.472112


Час виконання функції: 0.11536150000756606 секунд


3.4. Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [11]:
def get_records_by_time(df):
    filtered_df = df[(df['Time'] > '18:00:00') & (df['Global_active_power'] > 6.0) & (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])]
    n = len(filtered_df)
    mid = n // 2
    
    first_half = filtered_df.iloc[:mid]
    second_half = filtered_df.iloc[mid:]

    selected_first = first_half.iloc[::3]
    selected_second = second_half.iloc[::4]
    
    return pd.concat([selected_first, selected_second])  

display(get_records_by_time(df).head()) 
time_for_execution = timeit.timeit(lambda: get_records_by_time(df), number=1)
print(f'Час виконання функції: {time_for_execution} секунд')

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17492,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17496,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17499,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


Час виконання функції: 0.09676230000331998 секунд


### 4: Завдання

Пронормувати та стандартизувати вибраний датасет

In [12]:
def scale_data(df):
    numeric_col = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    
    data = df[numeric_col]
    
    print("Пронормовані дані:")
    normalized = pd.DataFrame(MinMaxScaler().fit_transform(data), columns=numeric_col)
    display(normalized.head())
    
    print("Стандартизовані дані:")
    standardized = pd.DataFrame(StandardScaler().fit_transform(data), columns=numeric_col)
    display(standardized.head())
    
    return normalized, standardized

normalized, standardized = scale_data(df)

Пронормовані дані:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


Стандартизовані дані:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955077,2.610721,-1.851816,3.098789,-0.182337,-0.051274,1.249421
1,4.037085,2.770406,-2.225274,4.133800,-0.182337,-0.051274,1.130897
2,4.050326,3.320432,-2.330213,4.133800,-0.182337,0.120487,1.249421
3,4.063567,3.355917,-2.191324,4.133800,-0.182337,-0.051274,1.249421
4,2.434881,3.586573,-1.592556,2.513782,-0.182337,-0.051274,1.249421


### 5: Завдання

Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [13]:
def analyze_correlation(df):

    pearson_val = df['Global_intensity'].corr(df['Global_active_power'], method='pearson')
    spearman_val = df['Global_intensity'].corr(df['Global_active_power'], method='spearman')
    
    print(f"Коефіцієнт Пірсона: {pearson_val}")
    print(f"Коефіцієнт Спірмена: {spearman_val}")

analyze_correlation(df)

Коефіцієнт Пірсона: 0.9988886002095827
Коефіцієнт Спірмена: 0.9953719367299748


### 6: Завдання

Провести One Hot Encoding категоріального атрибута.

In [23]:
def encode_season(df):
    df['Season'] = pd.to_datetime(df['Date'], dayfirst=True).dt.month.map({
        12:'Winter', 1:'Winter', 2:'Winter',
        3:'Spring', 4:'Spring', 5:'Spring',
        6:'Summer', 7:'Summer', 8:'Summer',
        9:'Autumn', 10:'Autumn', 11:'Autumn'
    })
    
    ohe = pd.get_dummies(df['Season'], prefix='Season')
    display(pd.concat([df[['Date', 'Season']], ohe], axis=1))

encode_season(df)

,Date,Season,Season_Autumn,Season_Spring,Season_Summer,Season_Winter
0,16/12/2006,Winter,False,False,False,True
1,16/12/2006,Winter,False,False,False,True
2,16/12/2006,Winter,False,False,False,True
3,16/12/2006,Winter,False,False,False,True
4,16/12/2006,Winter,False,False,False,True
...,...,...,...,...,...,...
2049275,26/11/2010,Autumn,True,False,False,False
2049276,26/11/2010,Autumn,True,False,False,False
2049277,26/11/2010,Autumn,True,False,False,False
2049278,26/11/2010,Autumn,True,False,False,False
